# Teaching a Unitree G1 to Walk: Reinforcement Learning from Scratch

In this notebook we train a walking policy for the Unitree G1 humanoid robot (29 joints), from nothing to a
robot that follows velocity commands: walk forward, sideways, turn, and stand still.

We use:
- **MuJoCo Warp**: the MuJoCo physics simulator running on the GPU, so thousands of robots train in parallel
- **mjlab**: a training framework built on it
- **Unitree's training setup** (`unitree_rl_mjlab`): the robot model, rewards and settings Unitree uses for the G1

What you will learn:
1. Checking the machine
2. Installing the tools
3. The robot: joints, motors and how they are controlled
4. The task: what the robot sees, what it controls, and what it is rewarded for
5. A short test run to measure training speed
6. Full training
7. Reading the training curves
8. Exporting the policy and watching it walk

Requirements: a Kaggle notebook with **GPU T4 x2** and **Internet on** (Settings panel on the right).

## Settings

Every choice for a run is set here, so the notebook can run from top to bottom without changes further down
(for example as a background run started with the Kaggle command-line tool). Change these values to run a
different experiment.

In [ ]:
# ---- Experiment settings ----
TASK = "Unitree-G1-Flat"   # Unitree's walking task on flat ground (chapter 4 explains it)
NUM_ENVS = 4096            # robots simulated in parallel on each GPU (Unitree's default)
ITERATIONS = 3000          # training iterations per run; each one simulates 24 steps x NUM_ENVS robots.
                           # 30 = quick pipeline test (~3 min), 3000 = baseline (~4.3 h on a T4)
SEEDS = (1, 2)             # one training run per seed; runs share out the GPUs (one run per GPU at a time)
RUN_TAG = "baseline"       # part of each run's folder name: logs/rsl_rl/g1_velocity/<date>_<RUN_TAG>_seed<n>
RUN_SMOKE_TEST = False     # chapter 5: 30-iteration speed test on one and two GPUs

## 1. Checking the machine

Before installing anything, we check what Kaggle gave us. Training needs:
- an **NVIDIA GPU**: the simulator runs on it (mjlab cannot train on a CPU or a Mac)
- **Python 3.10-3.13** and **PyTorch 2.7 or newer**: required by mjlab 1.2.0
- a GPU driver that supports the CUDA version PyTorch was built for

The cell prints the GPU(s), their memory and compute capability (the chip generation; a T4 is 7.5),
and the Python and PyTorch versions.

In [ ]:
import sys
import torch

# The GPU as the driver sees it. Top right: the newest CUDA version the driver supports.
!nvidia-smi

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__} (built for CUDA {torch.version.cuda})")
print(f"CUDA usable  : {torch.cuda.is_available()}")
print(f"GPU count    : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    memory_gb = torch.cuda.get_device_properties(i).total_memory / 2**30
    print(f"  GPU {i}      : {torch.cuda.get_device_name(i)}, "
          f"compute capability {major}.{minor}, {memory_gb:.1f} GB")

# Simulation and training packages already installed (none yet: chapter 2 installs them)
!pip list 2>/dev/null | grep -i -E "^(torch|mujoco|warp|mjlab) "

**Result:** two Tesla T4 GPUs with 15 GB each (compute capability 7.5), Python 3.13 and PyTorch 2.11
built for CUDA 12.8, with a driver that supports CUDA up to 13.0. Everything mjlab needs is already here
except the simulator and mjlab itself, which we install next. PyTorch does not need to be replaced.

## 2. Installing the tools

Two pieces:
- **mjlab** (with MuJoCo Warp): the GPU simulator and training framework. MuJoCo Warp is MuJoCo's physics running
  on the GPU: it simulates thousands of independent copies of the robot in a single step, which is what makes
  reinforcement learning fast enough to be practical.
- **Unitree's training code** (`unitree_rl_mjlab`): the G1 robot model and the walking task, with its rewards
  and settings

Installing Unitree's code pulls in the versions it pins (mjlab 1.2.0, MuJoCo Warp 3.5.0). Two more need pinning by
hand: `mujoco==3.5.0` and `warp-lang==1.12.0`. Their newest releases no longer match mjlab 1.2.0 and MuJoCo Warp
3.5.0 (they fail with `mjENBL_MULTICCD` and `module 'warp' has no attribute 'context'`), so we use the releases
from the same time as mjlab 1.2.0. We also check out a fixed commit of Unitree's code. Pinning every version is
what keeps a training setup reproducible.

The repository is cloned to `/tmp`: only `/kaggle/working` is saved as the notebook's output, and the repository
(about 490 MB) doesn't need saving. Training writes into the repository's `logs/` folder, so we link that folder to
`/kaggle/working/logs`: logs, checkpoints and the exported policy end up in the output, even if a run stops early.

The check below prints the installed versions, the GPUs Warp can use, and the G1 training tasks we can train.

In [ ]:
# Unitree's training code, at a fixed commit so this notebook stays reproducible
![ -d /tmp/unitree_rl_mjlab ] || git clone -q https://github.com/unitreerobotics/unitree_rl_mjlab.git /tmp/unitree_rl_mjlab
%cd /tmp/unitree_rl_mjlab
!git checkout -q 1425b15

# Training writes to logs/ inside the repository: make that a link to the saved output folder
!mkdir -p /kaggle/working/logs && ln -sfn /kaggle/working/logs logs

# Unitree's package ("src") with the versions it pins (mjlab 1.2.0, mujoco-warp 3.5.0), plus the two pins it
# misses (see above). PyTorch is already new enough, so pip keeps the preinstalled one.
%pip install -q -e . "mujoco==3.5.0" "warp-lang==1.12.0"

In [ ]:
from importlib.metadata import version

import warp
from mjlab.tasks.registry import list_tasks

for package in ("mujoco", "mujoco-warp", "warp-lang", "mjlab", "torch"):
    print(f"{package:12s}: {version(package)}")

warp.init()          # prints the devices Warp can use: both T4s should appear as CUDA devices

import src.tasks     # registers Unitree's training tasks (as their train.py does)
print("Tasks:", [t for t in list_tasks() if "G1" in t])

## 5. A short test run: how fast does it train?

Thirty training iterations on one GPU, then on two. Only runs when `RUN_SMOKE_TEST = True` (Settings).

What we measured on Kaggle's T4s (4096 robots per GPU): about **5.1 s per iteration** (4.6 s simulating, 0.5 s
updating the network), about 19,000 simulation steps per second per GPU. The first run also takes about two minutes
to start, mostly MuJoCo Warp compiling its GPU code; later runs reuse the compiled code. With two GPUs each one
simulates its own 4096 robots in the same time, so an iteration collects twice the experience.

In [ ]:
import torch

if RUN_SMOKE_TEST:
    # One GPU (the default). All output goes to a log file; we show only its end.
    !python scripts/train.py {TASK} --env.scene.num-envs {NUM_ENVS} --agent.max-iterations 30 \
        --agent.logger tensorboard --agent.run-name smoke_1gpu > /kaggle/working/smoke_1gpu.log 2>&1
    !tail -n 40 /kaggle/working/smoke_1gpu.log

    # Two GPUs, one training run spread over both (mjlab's list format for --gpu-ids)
    if torch.cuda.device_count() >= 2:
        !python scripts/train.py {TASK} --env.scene.num-envs {NUM_ENVS} --agent.max-iterations 30 \
            --agent.logger tensorboard --gpu-ids "[0, 1]" --agent.run-name smoke_2gpu > /kaggle/working/smoke_2gpu.log 2>&1
        !tail -n 40 /kaggle/working/smoke_2gpu.log

## 6. Training

One training run per seed in `SEEDS`, each on its own GPU, at the same time. With two T4s, two seeds train in the
time of one. With fewer GPUs than seeds, the runs take turns.

Why one run per GPU rather than one run over both GPUs: a two-GPU run collects twice the robots per update, which
changes the training recipe. Keeping Unitree's one-GPU recipe means every run, and every experiment later, is
trained the same way, and the two GPUs simply finish two runs at once. Two seeds of the same setup also show how much
results vary just from randomness, which we need before comparing different setups.

Each run's output goes to `/kaggle/working/train_<RUN_TAG>_seed<n>.log`, and its checkpoints (every 100
iterations) and exported policy to `logs/rsl_rl/g1_velocity/<date>_<RUN_TAG>_seed<n>/`.

In [ ]:
import subprocess
import torch

num_gpus = torch.cuda.device_count()
if num_gpus == 0:
    raise RuntimeError("No GPU: training needs one. Check the notebook's accelerator setting (GPU T4 x2).")
print(f"{num_gpus} GPU(s), {len(SEEDS)} run(s) of {ITERATIONS} iterations")


def train_command(seed, gpu):
    """The train.py command for one run: Unitree's settings, our iteration count and seed, on one GPU."""
    return ["python", "scripts/train.py", TASK,
            "--env.scene.num-envs", str(NUM_ENVS),
            "--agent.max-iterations", str(ITERATIONS),
            "--agent.seed", str(seed),
            "--agent.logger", "tensorboard",            # default is wandb, which would ask for a login
            "--agent.run-name", f"{RUN_TAG}_seed{seed}",
            "--gpu-ids", f"[{gpu}]"]                    # mjlab then shows this run only that GPU


# Start up to num_gpus runs at once (one per GPU), wait for them, then start the next group
for first in range(0, len(SEEDS), num_gpus):
    running = []
    for gpu, seed in enumerate(SEEDS[first:first + num_gpus]):
        log = open(f"/kaggle/working/train_{RUN_TAG}_seed{seed}.log", "w")
        process = subprocess.Popen(train_command(seed, gpu), stdout=log, stderr=subprocess.STDOUT)
        running.append((seed, gpu, process, log))
        print(f"started seed {seed} on GPU {gpu}")
    for seed, gpu, process, log in running:
        process.wait()
        log.close()
        print(f"seed {seed} (GPU {gpu}) finished with exit code {process.returncode} (0 = success)")

# The end of each log: the last iteration's statistics, or the error if a run failed
for seed in SEEDS:
    print(f"\n===== seed {seed} =====")
    !tail -n 30 /kaggle/working/train_{RUN_TAG}_seed{seed}.log

## What this run produced

Everything below `/kaggle/working` is saved as the notebook's output and can be downloaded: the training logs
(TensorBoard files for the curves in chapter 7), checkpoints `model_<iteration>.pt` and the exported policy
`.onnx` (chapter 8).

In [ ]:
!find /kaggle/working/logs/ \( -name "*.onnx" -o -name "model_*.pt" -o -name "events.out.*" \) | sort
!du -sh /kaggle/working/logs